# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/12-kartik66/flyrank-ml-internship/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
# Setup: imports and DuckDB connection to warehouse
%pip -q install duckdb huggingface_hub python-dotenv

import duckdb
import os
from dotenv import load_dotenv
# Load .env file for local execution (HF_TOKEN stored there, never in repo)
load_dotenv()
# Read HF_TOKEN from environment (set as Colab Secret or env var)
# Never paste a token into a cell in a public repo
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("HF_TOKEN not set. Add it as a Colab Secret or environment variable.")

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{hf_token}')")

# Base paths for the warehouse tables
BASE = "hf://datasets/FlyRank/internship-warehouse"
FACT_DAILY = f"{BASE}/fact_content_daily_performance"
FACT_DAILY_SAMPLE = f"{BASE}/fact_content_daily_performance_sample.parquet"
DIM_CLIENTS = f"{BASE}/dim_clients"
DIM_CONTENT = f"{BASE}/dim_content"
FACT_QUERY_90D = f"{BASE}/fact_content_query_90d"

# Quick sanity check: count and date range on the SAMPLE table (flat, not partitioned)
print("=== fact_content_daily_performance_sample (final month: June 2026) ===")
full_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date
    FROM read_parquet('{FACT_DAILY_SAMPLE}')
""").df()
print(full_check)

# Check the available month partitions in the main fact table
print("\n=== Available month partitions ===")
months = con.sql(f"""
    SELECT DISTINCT month
    FROM read_parquet('{FACT_DAILY}/month=*/*.parquet')
    ORDER BY month
""").df()
print(months)


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.


=== fact_content_daily_performance_sample (final month: June 2026) ===


   total_rows   min_date   max_date
0    11694072 2026-06-01 2026-06-30

=== Available month partitions ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

      month
0   2025-01
1   2025-02
2   2025-03
3   2025-04
4   2025-05
5   2025-06
6   2025-07
7   2025-08
8   2025-09
9   2025-10
10  2025-11
11  2025-12
12  2026-01
13  2026-02
14  2026-03
15  2026-04
16  2026-05
17  2026-06


## 1. Unit of analysis + time window

**One row = one content-item-day** (a specific page on a specific calendar date), drawn from `fact_content_daily_performance` partitioned by `month=YYYY-MM`.

**Table(s) used:** `fact_content_daily_performance` (primary), joined with `dim_content` for content metadata and `dim_clients` for client history coverage flags.

**Time window for development:** `month=2026-03` (a mid-panel month, not the final month). This gives ~31 days of daily rows per content item that existed in March 2026.

**What I'd predict/rank (label or proxy):** Whether a content item's **impressions will decline in the next 30-day window** relative to the current 30-day window — i.e., a forward-looking `trend_direction == "down"` computed from future daily impressions. The proxy label would be computed from daily data *after* the feature window closes.

**One thing I deliberately exclude:** `ga4_*` metrics (sessions, pageviews, engaged_sessions, ai_sessions, scroll_events) for dates before the client's `ga4_data_start` — these are zero-filled with `ga4_data_available = FALSE` and do not represent real engagement. I also exclude `gsc_avg_position = 0` rows (meaning "no position data") from position-based features.

In [2]:
# Verification for Section 1: inspect the March 2026 partition
MONTH = "2026-03"
FACT_MONTH = f"{FACT_DAILY}/month={MONTH}/*.parquet"

print(f"=== fact_content_daily_performance for {MONTH} ===")
mar_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT content_hash_id) AS n_content,
        COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{FACT_MONTH}')
""").df()
print(mar_check)

# Column list for reference
print("\n=== Columns in fact_content_daily_performance ===")
cols = con.sql(f"SELECT * FROM read_parquet('{FACT_MONTH}') LIMIT 0").df()
print(cols.columns.tolist())

=== fact_content_daily_performance for 2026-03 ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   total_rows   min_date   max_date  n_content  n_clients
0     9841378 2026-03-01 2026-03-31     331437         55

=== Columns in fact_content_daily_performance ===


['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


## 2. Fields: feature / label / context / excluded

**Feature (knowable BEFORE prediction moment):**
- `gsc_impressions` (daily GSC impressions)
- `gsc_clicks` (daily GSC clicks)
- `gsc_avg_position` (daily average position, excluding 0="no data")
- `gsc_data_available` (flag — knowable at decision time)
- `ga4_data_available` (flag — knowable at decision time)
- `report_date` (the decision date itself)
- From `dim_content`: `word_count`, `content_type`, `main_intent`, `search_volume`, `competition`, `cpc`, `content_age_days` (at snapshot), `days_since_last_update`
- From `dim_clients`: `gsc_data_start`, `ga4_data_start` (for window alignment)

**Label / Proxy (the target, NEVER a feature):**
- Forward 30-day `trend_direction` computed from daily impressions *after* the feature window (e.g., April 2026 impressions vs March 2026 impressions). This is the `is_declining_label` analogue.

**Context (for grouping/joining/splitting, never for model learning):**
- `content_hash_id`, `client_hash_id`
- `report_date` (as a partition key)

**Excluded (with one-line why):**
- `ga4_sessions`, `ga4_pageviews`, `ga4_engaged_sessions`, `ga4_ai_sessions`, `ga4_scroll_events` when `ga4_data_available IS NOT TRUE` — these are zero-filled before GA4 linkage, not real zeros.
- `gsc_avg_position` when `= 0` — means "no position data", not rank 0.
- `gsc_clicks`, `gsc_impressions` from the *future* (post-window) — leakage.
- `trend_pct`, `trend_direction` (or any forward-computed trend) — these *are* the label construction.
- `provider_used`, `model_used` — content generation metadata, not search performance signals.

In [3]:
# Verification for Section 2: inspect dim_content and dim_clients columns
print("=== dim_content columns ===")
dim_content_cols = con.sql(f"SELECT * FROM read_parquet('{DIM_CONTENT}.parquet') LIMIT 0").df()
print(dim_content_cols.columns.tolist())

print("\n=== dim_clients columns ===")
dim_clients_cols = con.sql(f"SELECT * FROM read_parquet('{DIM_CLIENTS}.parquet') LIMIT 0").df()
print(dim_clients_cols.columns.tolist())

=== dim_content columns ===


['client_hash_id', 'content_hash_id', 'keyword_hash_id', 'url_hash_id', 'keyword_char_count', 'keyword_token_count', 'url_char_count', 'content_created_date', 'content_updated_date', 'content_type', 'search_volume', 'competition', 'competition_level', 'cpc', 'main_intent', 'backlinks', 'category_count', 'keyword_created_date', 'provider_used', 'model_used', 'char_count', 'word_count', 'last_optimized_date', 'optimization_eligible_date', 'is_published', 'is_deleted']

=== dim_clients columns ===


['client_hash_id', 'is_active', 'has_gsc_access', 'has_ga4_access', 'access_profile', 'client_created_date', 'client_updated_date', 'gsc_data_start', 'ga4_data_start']


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [4]:
# QUERY 1 — Grain check: one row per content_hash_id × report_date?
# Should return ZERO rows if grain holds.
MONTH = "2026-03"
FACT_MONTH = f"{FACT_DAILY}/month={MONTH}/*.parquet"

grain_check = con.sql(f"""
    SELECT content_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{FACT_MONTH}')
    GROUP BY content_hash_id, report_date
    HAVING COUNT(*) > 1
    LIMIT 5
""").df()
print("=== Grain check (should be empty) ===")
print(grain_check)
print(f"Rows with duplicate grain: {len(grain_check)}")

=== Grain check (should be empty) ===
Empty DataFrame
Columns: [content_hash_id, report_date, c]
Index: []
Rows with duplicate grain: 0


In [5]:
# QUERY 2 — Counts & date span for the slice
counts_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        MIN(report_date) AS min_date,
        MAX(report_date) AS max_date,
        COUNT(DISTINCT content_hash_id) AS n_content,
        COUNT(DISTINCT client_hash_id) AS n_clients
    FROM read_parquet('{FACT_MONTH}')
""").df()
print("=== Counts & date span for 2026-03 ===")
print(counts_check)

=== Counts & date span for 2026-03 ===
   total_rows   min_date   max_date  n_content  n_clients
0     9841378 2026-03-01 2026-03-31     331437         55


In [6]:
# QUERY 3 — Availability: filter with IS TRUE on ga4_data_available and gsc_data_available
# Show how many rows survive the availability filter
avail_check = con.sql(f"""
    SELECT 
        COUNT(*) AS total_rows,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN 1 ELSE 0 END) AS gsc_available_true,
        SUM(CASE WHEN gsc_data_available IS FALSE THEN 1 ELSE 0 END) AS gsc_available_false,
        SUM(CASE WHEN gsc_data_available IS NULL THEN 1 ELSE 0 END) AS gsc_available_null,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_true,
        SUM(CASE WHEN ga4_data_available IS FALSE THEN 1 ELSE 0 END) AS ga4_available_false,
        SUM(CASE WHEN ga4_data_available IS NULL THEN 1 ELSE 0 END) AS ga4_available_null
    FROM read_parquet('{FACT_MONTH}')
""").df()
print("=== Availability flag breakdown (IS TRUE / IS FALSE / IS NULL) ===")
print(avail_check)

# Rows surviving the IS TRUE filter for BOTH GSC and GA4
survivors = con.sql(f"""
    SELECT COUNT(*) AS rows_both_available
    FROM read_parquet('{FACT_MONTH}')
    WHERE gsc_data_available IS TRUE
      AND ga4_data_available IS TRUE
""").df()
print(f"\nRows with gsc_data_available IS TRUE AND ga4_data_available IS TRUE: {survivors.iloc[0,0]}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== Availability flag breakdown (IS TRUE / IS FALSE / IS NULL) ===
   total_rows  gsc_available_true  gsc_available_false  gsc_available_null  \
0     9841378           3611061.0            6230317.0                 0.0   

   ga4_available_true  ga4_available_false  ga4_available_null  
0            413966.0            6408671.0           3018741.0  



Rows with gsc_data_available IS TRUE AND ga4_data_available IS TRUE: 364347


## 3 (cont.). Five features + "available when" justification

*Build a small feature frame for your lane from month=2026-03, and give every feature one line: "knowable at the decision moment because…"*

In [7]:
# Build 5-feature frame for March 2026, aggregated to content-item level
# Features are aggregated from DAILY rows to per-content features knowable at end of March
MONTH = "2026-03"
FACT_MONTH = f"{FACT_DAILY}/month={MONTH}/*.parquet"

feature_frame = con.sql(f"""
    SELECT 
        f.content_hash_id,
        f.client_hash_id,
        -- Feature 1: Total impressions in the month (knowable at decision moment because
        -- the month has ended and GSC data is finalized before we predict)
        SUM(f.gsc_impressions) AS feat_impressions_30d,
        -- Feature 2: Total clicks in the month (knowable at decision moment because
        -- clicks are recorded daily in GSC and finalized within days)
        SUM(f.gsc_clicks) AS feat_clicks_30d,
        -- Feature 3: Average position (excluding 0='no data') (knowable at decision moment because
        -- GSC position is reported daily and available when the month closes)
        AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END) AS feat_avg_position,
        -- Feature 4: GSC data availability rate (knowable at decision moment because
        -- the flag is recorded per day and we know which days had GSC access)
        AVG(CASE WHEN f.gsc_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS feat_gsc_availability_rate,
        -- Feature 5: GA4 data availability rate (knowable at decision moment because
        -- the flag is recorded per day; we only trust GA4 metrics on TRUE days)
        AVG(CASE WHEN f.ga4_data_available IS TRUE THEN 1.0 ELSE 0.0 END) AS feat_ga4_availability_rate
    FROM read_parquet('{FACT_MONTH}') f
    WHERE f.gsc_data_available IS TRUE  -- only trust GSC rows
    GROUP BY f.content_hash_id, f.client_hash_id
""").df()

print("=== 5-Feature frame (per content item, March 2026) ===")
print(f"Shape: {feature_frame.shape}")
print(feature_frame.head(10))
print("\n=== Feature statistics ===")
print(feature_frame.describe())

# Feature availability justifications (printed for the contract)
print("\n=== Feature availability justifications ===")
justifications = [
    "feat_impressions_30d: knowable at the decision moment because the calendar month has ended and GSC impressions are finalized before prediction.",
    "feat_clicks_30d: knowable at the decision moment because daily GSC clicks are recorded and stable within days of month-end.",
    "feat_avg_position: knowable at the decision moment because daily average position (excluding 0='no data') is reported by GSC and available when the month closes.",
    "feat_gsc_availability_rate: knowable at the decision moment because the gsc_data_available flag is recorded per day and we observe the full month's coverage.",
    "feat_ga4_availability_rate: knowable at the decision moment because the ga4_data_available flag is recorded per day; we only trust GA4 metrics on TRUE days."
]
for j in justifications:
    print(f"- {j}")

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

=== 5-Feature frame (per content item, March 2026) ===
Shape: (176738, 7)
            content_hash_id           client_hash_id  feat_impressions_30d  \
0  content_c59d55cfb9bfc8ef  client_62f4a7e64f5e0096                 605.0   
1  content_0ce079da38a7d96f  client_62f4a7e64f5e0096               15970.0   
2  content_31d6ccc62d1e9a44  client_62f4a7e64f5e0096                 146.0   
3  content_a0143ab7183a83e7  client_62f4a7e64f5e0096                 214.0   
4  content_3bba728db3a7140a  client_62f4a7e64f5e0096                  96.0   
5  content_1dc34d41453861ba  client_62f4a7e64f5e0096                  91.0   
6  content_7221ace46422f15f  client_62f4a7e64f5e0096                2052.0   
7  content_55472a9b756a23c1  client_62f4a7e64f5e0096                1395.0   
8  content_bcb2a914037f4d32  client_62f4a7e64f5e0096                  95.0   
9  content_5e1984a2c9185c02  client_62f4a7e64f5e0096                  47.0   

   feat_clicks_30d  feat_avg_position  feat_gsc_availability_rate  

## 3 (cont.). The trap — deliberate leakage experiment

*Add ONE label-derived column on purpose, watch your quick score jump toward perfect, then delete it and keep the honest number — the leakage lesson from notebook 02, performed on real warehouse data by you.*

In [8]:
# LEAKAGE EXPERIMENT: Add a label-derived feature and see AUC jump
# We'll simulate a "future trend" label using April 2026 data (next month)
# and show how using it as a "feature" inflates performance.

# First, get April 2026 data to compute a forward label
APRIL_MONTH = "2026-04"
FACT_APRIL = f"{FACT_DAILY}/month={APRIL_MONTH}/*.parquet"

# Compute forward trend label: April impressions vs March impressions per content_hash_id
print("=== Computing forward label from April 2026 (LEAKAGE SOURCE) ===")
forward_label = con.sql(f"""
    WITH mar AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS mar_impressions
        FROM read_parquet('{FACT_MONTH}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    ),
    apr AS (
        SELECT content_hash_id, SUM(gsc_impressions) AS apr_impressions
        FROM read_parquet('{FACT_APRIL}')
        WHERE gsc_data_available IS TRUE
        GROUP BY content_hash_id
    )
    SELECT 
        m.content_hash_id,
        m.mar_impressions,
        a.apr_impressions,
        CASE 
            WHEN m.mar_impressions = 0 THEN NULL
            WHEN (a.apr_impressions - m.mar_impressions) * 1.0 / m.mar_impressions < -0.2 THEN 1
            ELSE 0
        END AS is_declining_forward
    FROM mar m
    JOIN apr a ON m.content_hash_id = a.content_hash_id
""").df()

print(f"Forward label computed for {len(forward_label)} content items")
print(f"Decline rate: {forward_label['is_declining_forward'].mean():.3f}")
print(forward_label.head())

# Merge the LEAKY feature (forward label) into our feature frame
feature_with_leak = feature_frame.merge(
    forward_label[['content_hash_id', 'is_declining_forward']], 
    on='content_hash_id', 
    how='left'
)

# Quick model test: without leak vs with leak
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import numpy as np

# Prepare data (drop NaN labels)
data = feature_with_leak.dropna(subset=['is_declining_forward']).copy()
X = data[['feat_impressions_30d', 'feat_clicks_30d', 'feat_avg_position',
          'feat_gsc_availability_rate', 'feat_ga4_availability_rate']].fillna(0)
y = data['is_declining_forward'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)

# Model WITHOUT leak (honest features only)
rf_honest = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_honest.fit(X_train, y_train)
y_pred_honest = rf_honest.predict_proba(X_test)[:, 1]
auc_honest = roc_auc_score(y_test, y_pred_honest)
print(f"\n=== AUC WITHOUT leak (honest features): {auc_honest:.4f} ===")

# Model WITH leak (adding the forward label as a "feature")
X_leak = X.copy()
X_leak['leaky_feature'] = data.loc[X.index, 'is_declining_forward'].values

X_train_l, X_test_l, y_train_l, y_test_l = train_test_split(X_leak, y, test_size=0.3, random_state=42, stratify=y)

rf_leaky = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf_leaky.fit(X_train_l, y_train_l)
y_pred_leaky = rf_leaky.predict_proba(X_test_l)[:, 1]
auc_leaky = roc_auc_score(y_test_l, y_pred_leaky)
print(f"=== AUC WITH leak (forward label as feature): {auc_leaky:.4f} ===")
print(f"\nAUC jump from leakage: {auc_leaky - auc_honest:.4f}")

# NOW DELETE THE LEAKY FEATURE — keep only honest features
print(f"\n>>> LEAKY FEATURE REMOVED. Honest AUC kept: {auc_honest:.4f} <<<")

# Show feature importance for honest model
import pandas as pd
imp = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_honest.feature_importances_
}).sort_values('importance', ascending=False)
print("\n=== Honest feature importances ===")
print(imp)

=== Computing forward label from April 2026 (LEAKAGE SOURCE) ===


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Forward label computed for 158549 content items
Decline rate: 0.478
            content_hash_id  mar_impressions  apr_impressions  \
0  content_7aab0418a7593b66              3.0              1.0   
1  content_2217c13531f2c094           1673.0           2348.0   
2  content_cff6e0d5a8234bc0           5373.0           4705.0   
3  content_0e034bf052f81ee8           6750.0           1083.0   
4  content_0d8b27cf2b8f7a61            235.0            116.0   

   is_declining_forward  
0                     1  
1                     0  
2                     0  
3                     1  
4                     1  



=== AUC WITHOUT leak (honest features): 0.6054 ===


=== AUC WITH leak (forward label as feature): 1.0000 ===

AUC jump from leakage: 0.3946

>>> LEAKY FEATURE REMOVED. Honest AUC kept: 0.6054 <<<

=== Honest feature importances ===
                      feature  importance
2           feat_avg_position    0.517299
0        feat_impressions_30d    0.343981
4  feat_ga4_availability_rate    0.084962
1             feat_clicks_30d    0.053758
3  feat_gsc_availability_rate    0.000000


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [9]:
# Investigate data limitations for this slice
print("=== Per-client history depth (gsc_data_start) ===")
client_history = con.sql(f"""
    SELECT client_hash_id, gsc_data_start, ga4_data_start
    FROM read_parquet('{DIM_CLIENTS}.parquet')
    ORDER BY gsc_data_start
""").df()
print(client_history.to_string(index=False))

# Check how many clients have data in March 2026 vs earlier
print("\n=== Clients active in March 2026 ===")
mar_clients = con.sql(f"""
    SELECT client_hash_id, COUNT(DISTINCT content_hash_id) AS n_content, COUNT(*) AS n_rows
    FROM read_parquet('{FACT_MONTH}')
    GROUP BY client_hash_id
    ORDER BY n_rows DESC
""").df()
print(mar_clients.to_string(index=False))

# Missingness in dim_content for key feature columns
print("\n=== Missingness in dim_content (key feature columns) ===")
missing = con.sql(f"""
    SELECT 
        COUNT(*) AS total,
        SUM(CASE WHEN word_count IS NULL THEN 1 ELSE 0 END) AS word_count_null,
        SUM(CASE WHEN search_volume IS NULL THEN 1 ELSE 0 END) AS search_volume_null,
        SUM(CASE WHEN competition IS NULL THEN 1 ELSE 0 END) AS competition_null,
        SUM(CASE WHEN cpc IS NULL THEN 1 ELSE 0 END) AS cpc_null,
        SUM(CASE WHEN content_type IS NULL THEN 1 ELSE 0 END) AS content_type_null,
        SUM(CASE WHEN main_intent IS NULL THEN 1 ELSE 0 END) AS main_intent_null
    FROM read_parquet('{DIM_CONTENT}.parquet')
""").df()
print(missing)

=== Per-client history depth (gsc_data_start) ===


         client_hash_id gsc_data_start ga4_data_start
client_9958f0a7ae1df715     2025-01-27     2025-10-29
client_ff644d8251367cbb     2025-01-27     2025-10-29
client_73cda7b4e4f265ea     2025-02-11     2026-03-24
client_fef1a8f436438636     2025-03-11     2026-03-06
client_62f4a7e64f5e0096     2025-06-07            NaT
client_b10cb2997d0c7c86     2025-06-18     2025-11-15
client_c182d11e4862a37d     2025-06-21     2026-02-20
client_65de48885f4ef01b     2025-06-21     2026-02-19
client_3197e6291363b4db     2025-06-29     2025-11-09
client_625b6439094e23e4     2025-07-01     2026-02-19
client_a2eeb8899886adde     2025-07-06     2026-02-19
client_d211cb07b9059bab     2025-07-07     2026-02-19
client_def0955f7a377868     2025-07-17     2026-02-19
client_08d2847f24cf89c1     2025-07-21     2026-02-19
client_8ae2bfb5aa1ffa1e     2025-07-28            NaT
client_8dbf3abdf07569e0     2025-07-29            NaT
client_0e1acc6cd57b0eba     2025-09-24     2026-02-19
client_795153d5b7850ccf     

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

         client_hash_id  n_content  n_rows
client_625b6439094e23e4      31887  988497
client_3ffa76342f366962      31108  904847
client_73cda7b4e4f265ea      29333  869640
client_08a6a72ff48e62c0      28278  851275
client_62f4a7e64f5e0096      25356  756660
client_65de48885f4ef01b      13781  426307
client_23a62021009f63c4      14621  423613
client_ba65e80a1116ae41      13239  410409
client_2b4306c3ed003f01      12126  375906
client_fef1a8f436438636      11223  335379
client_3197e6291363b4db      10690  316610
client_e547b89c05043229       9308  288548
client_a80fca3f171ed1de       7626  235505
client_19b89ee4fe3db6da       6871  213001
client_f623b01661d4bfe4       5878  182218
client_2094c6eb080311d5       6375  173700
client_3f0ce4d44fe94f3d       6417  170957
client_a60a11451483af1c       5457  169167
client_cd12bcfd98942aa1       5663  159948
client_ff644d8251367cbb       4522  140182
client_20259bd6705d81d4       4806  136720
client_157ffe4d4a595515       4789  133632
client_b10c

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

    total  word_count_null  search_volume_null  competition_null  cpc_null  \
0  519606         177768.0            142622.0          142622.0  142622.0   

   content_type_null  main_intent_null  
0                0.0          148398.0  


## 4 (cont.). Named limitation of this slice

**Limitation: Unbalanced panel with GSC-only early history.**

- Clients have wildly different history start dates (`gsc_data_start` ranges from 2025-01-27 to 2026-04-01). March 2026 is only observable for clients whose `gsc_data_start <= 2026-03-01`.
- For clients with `ga4_data_start > 2026-03-01`, all GA4 metrics in March are zero-filled with `ga4_data_available = FALSE` — they contain no real engagement signal.
- 10 of 104 clients have `NULL` in `ga4_data_available` / `gsc_data_available` flags in `dim_clients`, meaning their availability status is unknown for the full history.
- The `fact_content_query_90d` table's 90-day window overlaps with March/April 2026 — using its `impressions_90d` or `*_last30` columns as features for a March→April label would leak the label period.
- Position `gsc_avg_position = 0` means "no data", not rank 0 (affects ~1.5% of rows in starter slice; similar proportion expected here).
- Content items registered mid-panel have incomplete daily history — the daily fact only accrues from registration day, while the query table backfills the full 90-day window.

## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.